# 01: Train the YOLOv8 defect detector

This notebook fine-tunes YOLOv8n on the pork rasher dataset (official split, `wrinkle` removed), then evaluates it on the validation and test sets. It saves everything needed for the analysis, so that part can run without a GPU.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Put the Roboflow zip `Pork_Rasher_Error-Packaging-_v4i_yolov8.zip` in your Google Drive, in a folder called `MMA3001`.
3. Run the cells in order (**Runtime → Run all** also works). Training takes about 30-45 minutes; keep this tab open.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive and find the dataset

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/MMA3001')
DATA_ZIP = DRIVE_DIR / 'Pork_Rasher_Error-Packaging-_v4i_yolov8.zip'
assert DATA_ZIP.exists(), f'Zip not found at {DATA_ZIP}. Check the folder and file name in Drive.'
print('Found', DATA_ZIP, f'({DATA_ZIP.stat().st_size/1e6:.0f} MB)')

## 3. Get the code and install YOLO

In [ ]:
%cd /content
!rm -rf 36137316-pork-rasher-inspection
!git clone -q https://github.com/mmal0078/36137316-pork-rasher-inspection
%cd /content/36137316-pork-rasher-inspection
!git log --oneline -1
!pip install -q ultralytics pytest

## 4. Prepare the data and run the tests

This unzips the export to local disk (much faster than reading from Drive), runs the audit, and writes `data/processed`. The audit printout should report **1317 distinct images**, **0 images in more than one split** and **3 conflicting labels**.

In [ ]:
!mkdir -p /content/data/raw && unzip -q -o "{DATA_ZIP}" -d /content/data/raw
!python scripts/prepare_data.py --raw /content/data/raw --out /content/data/processed | head -12
!python -m pytest -q

## 5. Train YOLOv8n

Settings:

| Setting | Value |
|---|---|
| Model | YOLOv8n, pretrained on COCO |
| Image size | 640 px |
| Batch | 16 |
| Epochs | up to 60, stopping early if validation mAP hasn't improved for 20 epochs |
| Seed | 0, deterministic |

Ultralytics' default augmentations are used (mosaic, flips, HSV). The epoch log prints as it goes.

In [ ]:
RUN_NAME = 'yolov8n_640'
!python scripts/train_yolo.py --data /content/data/processed/data.yaml --model yolov8n.pt \
    --epochs 60 --imgsz 640 --batch 16 --device 0 --out /content/runs --name {RUN_NAME}

## 6. Quick look at the results

In [ ]:
import json
m = json.load(open(f'/content/runs/{RUN_NAME}/metrics.json'))
print(f"Training time: {m['train_minutes']:.1f} min")
for split in ('val', 'test'):
    print(f"\n{split}: mAP50 = {m[split]['mAP50']:.3f}, mAP50-95 = {m[split]['mAP50_95']:.3f}")
    for c, v in m[split]['per_class'].items():
        print(f"   {c:16s} mAP50 = {v['mAP50']:.3f}")
from IPython.display import Image, display
display(Image(f'/content/runs/{RUN_NAME}/results.png', width=900))

## 7. Save the results to Drive

This zips the run folder (metrics, predictions, plots and weights, about 15 MB) and copies it to `MMA3001` in your Drive. **Download that zip and attach it in the chat with Claude.**

In [ ]:
import shutil
z = shutil.make_archive(f'/content/{RUN_NAME}', 'zip', '/content/runs', RUN_NAME)
shutil.copy(z, DRIVE_DIR / f'{RUN_NAME}.zip')
print('Saved', DRIVE_DIR / f'{RUN_NAME}.zip')